In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

file_path = '/content/drive/MyDrive/IAPD/Proiect_IAPD/dataset_anomaly.csv'
df = pd.read_csv(file_path)

print("Dimensiune dataset:", df.shape)
display(df.head())

Stergem coloanele care nu ajuta la predictie.

In [ ]:
print(df.columns.tolist())

In [ ]:
display(
    df[
        (df["capacity"] >= 5) &
        (~df["unit_description"].str.contains("entire", case=False, na=False))
    ]
    .sort_values(by="capacity", ascending=False)[
        ["name", "capacity", "url", "unit_description",
         "bedrooms", "price_per_night"]
    ]
)

In [ ]:
df = df.drop(index=[1374, 1401, 561]).copy()

print("Dimensiune dataset:", df.shape)

In [ ]:
null_counts = df.isna().sum()

print("Total valori lipsa:", null_counts.sum())

display(
    null_counts[null_counts > 0]
    .sort_values(ascending=False)
    .to_frame(name="Valori lipsa")
)

In [ ]:
display(
    df[df["bedrooms"].isna()][
        ["name", "city", "property_type", "bedrooms",
         "capacity", "url", "unit_description"]
    ]
)

In [ ]:
df = df.drop(index=[294]).copy()

print("Dimensiune dataset:", df.shape)

In [ ]:
null_counts = df.isna().sum()

print("Total valori lipsa:", null_counts.sum())

display(
    null_counts[null_counts > 0]
    .sort_values(ascending=False)
    .to_frame(name="Valori lipsa")
)

In [ ]:
coloane_de_sters = [
    "property_id",
    "name",
    "url",
    "unit_description",
    "checkin",
    "scraped_at",
    "distance_from_centre"
]

df = df.drop(columns=coloane_de_sters)

print("Coloane ramase:")
print(df.columns.tolist())

print("Dimensiune dataset:", df.shape)

In [ ]:
# Salvam datasetul filtrat
output_path = '/content/drive/MyDrive/IAPD/Proiect_IAPD/dataset_final.csv'

df.to_csv(output_path, index=False, encoding='utf-8-sig')

Rezolvam anomaliile de pret

In [ ]:
import numpy as np
import pandas as pd

PATH = '/content/drive/MyDrive/IAPD/Proiect_IAPD/dataset_final.csv'
df = pd.read_csv(PATH)
df = df.reset_index().rename(columns={'index': 'idx'})

df['pret_per_persoana'] = df['price_per_night'] / df['capacity']
df['pret_per_camera']   = df['price_per_night'] / df['bedrooms']
df['log_pret']          = np.log(df['price_per_night'])
df['log_pp_persoana']   = np.log(df['pret_per_persoana'])
df['log_pp_camera']     = np.log(df['pret_per_camera'])

df['cap_bin'] = pd.cut(df['capacity'], bins=[0, 2, 4, 6, 10, 100],
                       labels=['1-2', '3-4', '5-6', '7-10', '11+'])

MIN_GRUP = 8

def robust_z(serie):
    med = serie.median()
    mad = (serie - med).abs().median()
    sigma = 1.4826 * mad
    if not np.isfinite(sigma) or sigma < 1e-9:
        return pd.Series(0.0, index=serie.index)
    return (serie - med) / sigma

def z_pe_segment(df, col, chei):
    z = df.groupby(chei, observed=True)[col].transform(robust_z)
    marime = df.groupby(chei, observed=True)[col].transform('size')
    z_fallback = df.groupby('property_type')[col].transform(robust_z)
    return z.where(marime >= MIN_GRUP, z_fallback)

df['z_pret_segment'] = z_pe_segment(df, 'log_pret', ['property_type', 'cap_bin'])
df['z_per_persoana'] = z_pe_segment(df, 'log_pp_persoana', ['property_type'])
df['z_per_camera']   = z_pe_segment(df, 'log_pp_camera', ['property_type'])
df['z_oras_tip']     = z_pe_segment(df, 'log_pret', ['city', 'property_type'])

df.loc[df['capacity'] >= 7, 'z_pret_segment'] = np.nan
zcols = ['z_pret_segment', 'z_per_persoana', 'z_per_camera']

df['regula_mica_cap_pret_mare'] = (df['capacity'] <= 2) & (df['price_per_night'] > 1500) & \
                                  (df['property_type'].isin(['Apartment', 'Guest house', 'Bed and breakfast', 'Hostel']))
df['regula_pp_persoana_mare']   = df['pret_per_persoana'] > 800
df['regula_hostel_mare']        = (df['property_type'] == 'Hostel') & (df['price_per_night'] > 400)

PRAG_Z = 3.5

df['nr_criterii_z'] = (df[zcols] > PRAG_Z).sum(axis=1)
df['nr_reguli']     = df[['regula_mica_cap_pret_mare', 'regula_pp_persoana_mare', 'regula_hostel_mare']].sum(axis=1)
df['z_max']         = df[zcols].max(axis=1)
df['scor_suspiciune'] = df['nr_criterii_z'] + df['nr_reguli']

suspecte = df[(df['scor_suspiciune'] >= 2) | (df['z_max'] > 5)].copy()
suspecte = suspecte.sort_values(['scor_suspiciune', 'z_max'], ascending=False)

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 500)

coloane_afisate = ['idx', 'city', 'property_type', 'capacity', 'bedrooms', 'rating', 'num_reviews',
                   'price_per_night', 'pret_per_persoana', 'pret_per_camera',
                   'z_pret_segment', 'z_per_persoana', 'z_per_camera', 'z_oras_tip',
                   'scor_suspiciune']

print(f'Total cazari: {len(df)} | Suspecte: {len(suspecte)} '
      f'({len(suspecte) / len(df):.1%})')
print('\nTop suspecte:\n')
display(suspecte[coloane_afisate].round(2))

print('\nMediana pretului:')
display(df.pivot_table(index='property_type', columns='cap_bin', values='price_per_night',
                       aggfunc='median', observed=True).round(0))

df['pret_suspect'] = df['idx'].isin(suspecte['idx']).astype(int)

am sters liniile cu anomalii in pret:

In [ ]:
import pandas as pd

idx_de_sters = [1382, 548, 1436, 430, 417, 410, 928, 1450, 1468, 672,
                665, 996, 1066, 471, 1113, 546, 465, 466, 420]

df_original = pd.read_csv(PATH)
df_curat = df_original.drop(index=idx_de_sters).reset_index(drop=True)

print(f'Inainte: {len(df_original)} | Dupa: {len(df_curat)} | Sterse: {len(idx_de_sters)}')

Stergem liniile cu anomalie in rating

In [ ]:
import numpy as np

masca_bug = np.isclose(df_curat['rating'] * 10, df_curat['num_reviews']) & (df_curat['rating'] < 8)

print(f'Randuri cu bug: {masca_bug.sum()}')
print(df_curat.loc[masca_bug, ['city', 'property_type', 'rating', 'num_reviews', 'price_per_night']])

df_curat = df_curat[~masca_bug].reset_index(drop=True)
print(f'Randuri ramase: {len(df_curat)}')

stergem nr de facilitati

In [ ]:
df_curat = df_curat.drop(columns=['num_facilities_listed'])

In [ ]:
coloane_de_sters = [
    "pret_per_persoana",
    "pret_per_camera",
    "log_pret",
    "log_pp_persoana",
    "log_pp_camera",
    "cap_bin",
    "z_pret_segment",
    "z_per_persoana",
    "z_per_camera",
    "z_oras_tip",
    "regula_mica_cap_pret_mare",
    "regula_pp_persoana_mare",
    "regula_hostel_mare",
    "nr_criterii_z",
    "nr_reguli",
    "z_max",
    "scor_suspiciune",
    "pret_suspect"
]

df_curat = df_curat.drop(columns=coloane_de_sters, errors="ignore")

print("Coloane ramase:")
print(df_curat.columns.tolist())

print("Dimensiune dataset:", df_curat.shape)

In [ ]:
# search pt doua persoana
# stergem proprietatile cu capacitate de o pers

capacity_1 = df_curat[df_curat["capacity"] == 1]
display(capacity_1)
df_curat = df_curat[df_curat["capacity"] != 1].copy()

In [ ]:
df_curat = df_curat.drop(index=707).copy()

In [ ]:
for col in df_curat.columns:
    if not pd.api.types.is_numeric_dtype(df_curat[col]):
        print(f"\n{col}")
        print("Valori distincte:", df_curat[col].nunique())
        print(df_curat[col].value_counts(dropna=False).head(10))

Encoding:

In [ ]:
df_curat = pd.get_dummies(
    df_curat,
    columns=["city", "property_type"],
    dtype=int
)
display(df_curat.head())

In [ ]:
output_path = "/content/drive/MyDrive/IAPD/Proiect_IAPD/dataset_final4.csv"

df_curat.to_csv(output_path, index=False)
